Import the Mlp class and the Protein Strings

In [59]:
import torch 
from torch import nn
import torch.nn.functional as F
from torch import optim
from protgen.models.mlp import MLP
from protgen.data import ProteinDataset
from protgen.data.make_tokens import TokenDataset
from torch.utils.data import DataLoader
from protgen.data import AA, FAMILIES, IGNORE_INDEX, special_tokens

Load the Data 

In [17]:
train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get train tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")

#Get validation tensors
X_val = val_data.tokens
y_val_labels = val_data.labels 
y_val_shifted = val_data.tokens_shifted

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])


Create the model and test it with a small input 

In [4]:
mlp = MLP()

X = torch.LongTensor([[0, 0, 1], [0, 1, 2], [1, 2, 3]])
Y = torch.LongTensor([1,2,3])

logits = mlp(X)
targets = Y

Loss = F.cross_entropy(logits, targets)
print(f"Logits: {logits}")
print(f"Targets: {targets}")
print(f"Loss: {Loss}")


Logits: tensor([[-0.2087,  0.2121,  0.0158,  0.1143, -0.0050, -0.2678,  0.0338, -0.0632,
         -0.1051, -0.1812,  0.1943,  0.0542, -0.1138,  0.0989,  0.3371, -0.1810,
          0.4174,  0.1678, -0.1642,  0.0520,  0.0707, -0.0726,  0.0115],
        [-0.0291, -0.1409, -0.1894,  0.4180, -0.1071, -0.4906,  0.2185,  0.0201,
         -0.2634, -0.1055,  0.0655,  0.1744, -0.0124, -0.0152, -0.1147, -0.1552,
          0.3453,  0.0017, -0.2566,  0.0629,  0.1139,  0.0737, -0.1030],
        [-0.4922,  0.3900,  0.1159,  0.2483,  0.1242,  0.0356,  0.2931, -0.1138,
         -0.0856,  0.0904,  0.3522,  0.0579,  0.2911, -0.0057,  0.3298, -0.0673,
          0.6834,  0.2098, -0.2323, -0.0601, -0.0445, -0.0585, -0.0900]],
       grad_fn=<AddmmBackward0>)
Targets: tensor([1, 2, 3])
Loss: 3.094010591506958


Optimizer Function

In [5]:
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

Accuracy Function

In [ ]:
def accuracy(ys, ts):
    # making a one-hot encoded vector of correct (1) and incorrect (0) predictions
    correct_prediction = torch.eq(torch.max(ys, 1)[1], torch.max(ts, 1)[1])
    # averaging the one-hot encoded vector
    return torch.mean(correct_prediction.float())

Overfitting over the small input test

In [6]:
for epoch in range(1000):
    logits = mlp(X)
    loss = F.cross_entropy(logits, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

Epoch 0: Loss = 3.0940
Epoch 100: Loss = 0.0742
Epoch 200: Loss = 0.0127
Epoch 300: Loss = 0.0057
Epoch 400: Loss = 0.0033
Epoch 500: Loss = 0.0021
Epoch 600: Loss = 0.0015
Epoch 700: Loss = 0.0011
Epoch 800: Loss = 0.0009
Epoch 900: Loss = 0.0007


(OUTDATED!) Format the tranning data and create the 3 Tokens sequences test 

In [44]:
#Add double padding in the begging of the sequence so the first input is [0, 0, 1]
X_train_pad = F.pad(X_train, (2, 0), "constant", 0)
#print(f"Padded train data shape: {X_train_pad.shape}")
#print(f"Padded train data: {X_train_pad}")
#print(f"Original shifted data shape: {y_train_shifted.shape}")  
#print(f"Shifted train data: {y_train_shifted}")

X_train_inputs = X_train_pad.unfold(dimension=1, size=3, step=1)
print(f"Unfolded train inputs shape: {X_train_inputs.shape}")
#print(f"Unfolded train inputs: {X_train_inputs}")

X_val_pad = F.pad(X_val, (2, 0), "constant", 0)
X_val_inputs = X_val_pad.unfold(dimension=1, size=3, step=1)


#for i in range(X_train_inputs.shape[1]):
#    if X_train_inputs[1, i, 2] != y_train_shifted[1, i]:
#        print(f"Target input pair at index {i}: Input: {X_train_inputs[1, i, :]}, Target: {y_train_shifted[1, i]}")

X_train_inputs = X_train_inputs.reshape(-1, 3)
y_train_shifted = y_train_shifted.reshape(-1)
X_val_inputs = X_val_inputs.reshape(-1, 3)
y_val_shifted = y_val_shifted.reshape(-1)
print(f"Reshaped train inputs shape: {X_train_inputs.shape}")
print(f"Reshaped train targets shape: {y_train_shifted.shape}")

train_Loader = DataLoader(list(zip(X_train_inputs, y_train_shifted)), batch_size=32, shuffle=True)
val_Loader = DataLoader(list(zip(X_val_inputs, y_val_shifted)), batch_size=32, shuffle=False)

Unfolded train inputs shape: torch.Size([16020, 129, 3])
Reshaped train inputs shape: torch.Size([2066580, 3])
Reshaped train targets shape: torch.Size([2066580])


Create Tokens Dataset

In [62]:
train_dataset = TokenDataset(train_data)
val_dataset = TokenDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

x_train_batch, targets_train_batch = next(iter(train_loader))
print(f"batch: x {x_train_batch.shape} {x_train_batch.dtype}, y {targets_train_batch.shape}")

True
True
batch: x torch.Size([32, 3]) torch.int64, y torch.Size([32])


In [ ]:
# number of training passses
num_epochs = 1000
# store loss and accuracy for information
train_losses, val_losses, train_accs, val_accs = [], [], [], []


# training loop
for e in range(num_epochs):
    
    # zeroize accumulated gradients in parameters
    optimizer.zero_grad()
    # predict by running forward pass
    tr_output = mlp(X_train_inputs)
    # compute cross entropy loss
    tr_loss = F.cross_entropy(tr_output, y_train_shifted, Ignore_index=-IGNORE_INDEX)
    # compute gradients given loss
    tr_loss.backward()
    # update the parameters given the computed gradients
    optimizer.step()
    train_acc = accuracy(tr_output, y_train_shifted)

    # store training loss
    train_losses.append(tr_loss.data.numpy())
    train_accs.append(train_acc)


    # predict with validation input
    val_output = mlp(X_val_inputs)
    # compute loss and accuracy
    val_loss = F.cross_entropy(val_output, y_val_shifted, Ignore_index=-IGNORE_INDEX)
    val_acc = accuracy(val_output, y_val_shifted)

    # store loss and accuracy
    val_losses.append(val_loss.data.numpy())
    val_accs.append(val_acc.data.numpy())

    if e % 100 == 0:
        print("Epoch %i, "
              "Train Cost: %0.3f"
              "\tVal Cost: %0.3f"
              "\t Val acc: %0.3f" % (e,
                                     train_losses[-1],
                                     val_losses[-1],
                                     val_accs[-1]))


NameError: name 'X_tr' is not defined